# Feature Selection Analysis for Random Forest

In [1]:
import pandas as pd

from pathlib import Path
import pandas as pd
import sys

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS
from models_helper import ModelTrainer
from dataviz_helper import plot_feature_selection_curve

In [2]:
# get the data
DATA_DIR = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed.parquet")
df_train_over = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed_oversampled.parquet")
df_val   = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


In [3]:
# split features and labels
label_cols = [c for c in df_train.columns if c.startswith("label_")]

X_train_over = df_train_over[ALL_FEATURE_KEYS]
y_train_over = df_train_over[label_cols]
X_val   = df_val[ALL_FEATURE_KEYS]
y_val   = df_val[label_cols]

print(f"Train (oversampled): {len(X_train_over)}, Val: {len(X_val)}")
print(f"Labels: {label_cols}")

Train (oversampled): 28756, Val: 8458
Labels: ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 1. Read Feature Importance ranking

In [4]:
fi_df = pd.read_json("../4_1_train ML models/feature_importance_random_forest.json", orient="records")
fi_df

,feature,importance
0,geom_layer_count,0.049578
1,geom_ratio_area_vol,0.039275
2,aabb_diagonal,0.039034
3,tfbb_ratio_02,0.035958
4,tfbb_extent_0,0.035602
...,...,...
68,mat_kalksandstein,0.000005
69,mat_putz,0.000002
70,mat_mörtel,0.000000
71,mat_foamglas,0.000000


## 2. Incremental feature selection loop

In [5]:
# loop through features in order of importance and evaluate model performance with increasing number of features
ordered_features = fi_df["feature"].tolist()
loop_results = []

for n in range(1, len(ordered_features) + 1):
    features_subset = ordered_features[:n]
    model = ModelTrainer("random_forest", label_cols)
    model.fit(X_train_over[features_subset], y_train_over)
    
    metrics_df = model.evaluate(X_val[features_subset], y_val)
    score = metrics_df.loc["mean", "f1_macro"]

    print(f"{n:2d} Features | Last Feature: {ordered_features[n-1]:35s} | F1-macro: {score:.4f}")
    loop_results.append({"n_features": n, "last_feature_added": ordered_features[n - 1], "f1_macro": score})

# save the loop results to a csv file
loop_df = pd.DataFrame(loop_results)
loop_df.to_csv("feature_selection_results_random_forest.csv", index=False)
loop_df

 1 Features | Last Feature: geom_layer_count                    | F1-macro: 0.3127
 2 Features | Last Feature: geom_ratio_area_vol                 | F1-macro: 0.4976
 3 Features | Last Feature: aabb_diagonal                       | F1-macro: 0.5408
 4 Features | Last Feature: tfbb_ratio_02                       | F1-macro: 0.5578
 5 Features | Last Feature: tfbb_extent_0                       | F1-macro: 0.5729
 6 Features | Last Feature: topo_max_face_area                  | F1-macro: 0.6052
 7 Features | Last Feature: aabb_max_z                          | F1-macro: 0.6601
 8 Features | Last Feature: tfbb_sphericity                     | F1-macro: 0.6536
 9 Features | Last Feature: topo_avg_face_area                  | F1-macro: 0.6791
10 Features | Last Feature: geom_z_max                          | F1-macro: 0.6658
11 Features | Last Feature: geom_centroid_z                     | F1-macro: 0.6685
12 Features | Last Feature: geom_z_min                          | F1-macro: 0.6776
13 F

,n_features,last_feature_added,f1_macro
0,1,geom_layer_count,0.3127
1,2,geom_ratio_area_vol,0.4976
2,3,aabb_diagonal,0.5408
3,4,tfbb_ratio_02,0.5578
4,5,tfbb_extent_0,0.5729
...,...,...,...
68,69,mat_kalksandstein,0.7031
69,70,mat_putz,0.7008
70,71,mat_mörtel,0.6990
71,72,mat_foamglas,0.7076


## 4. Plot F1-macro vs. number of features

In [6]:
# plot the feature selection curve
# optimal top n features are 9
plot_feature_selection_curve(loop_df)